In [1]:
import pandas as pd

Carregando arquivos e concatenando em um único DataFrame

In [2]:
arquivos = [
    "Dublin Bikes Historical Station Data January 2026.csv",
    "Dublin Bikes Historical Station Data February 2026.csv",
    "Dublin Bikes Historical Station Data March 2026.csv",
    "Dublin Bikes Historical Station Data April 2026.csv",
    "Dublin Bikes Historical Station Data May 2026.csv",
    "Dublin Bikes Historical Station Data June 2026.csv",]
lista_dfs = []
for nome in arquivos:
    df_mes = pd.read_csv(f"../data/raw/{nome}")
    lista_dfs.append(df_mes)

df_completo = pd.concat(lista_dfs, ignore_index=True)
df_completo


,system_id,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,short_name,address,lat,lon,region_id,capacity
0,dublin_bikes,2026-01-01 00:05:00,1,24,7,True,True,True,CLARENDON ROW,NaN,Clarendon Row,53.340927,-6.262501,NaN,31
1,dublin_bikes,2026-01-01 00:05:00,10,16,0,True,True,True,DAME STREET,NaN,Dame Street,53.344006,-6.266802,NaN,16
2,dublin_bikes,2026-01-01 00:05:00,101,5,25,True,True,True,KING STREET NORTH,NaN,King Street North,53.350292,-6.273507,NaN,30
3,dublin_bikes,2026-01-01 00:05:00,103,5,35,True,True,True,GRANGEGORMAN LOWER (SOUTH),NaN,Grangegorman Lower (South),53.354664,-6.278681,NaN,40
4,dublin_bikes,2026-01-01 00:05:00,104,20,20,True,True,True,GRANGEGORMAN LOWER (CENTRAL),NaN,Grangegorman Lower (Central),53.355170,-6.278424,NaN,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3412085,dublin_bikes,2026-06-30 23:55:00,68,20,20,True,True,True,HANOVER QUAY,NaN,Hanover Quay,53.344116,-6.237153,NaN,40
3412086,dublin_bikes,2026-06-30 23:55:00,80,15,25,True,True,True,ST JAMES HOSPITAL (LUAS),NaN,St James Hospital (Luas),53.341360,-6.292951,NaN,40
3412087,dublin_bikes,2026-06-30 23:55:00,84,18,12,True,True,True,BROOKFIELD ROAD,NaN,Brookfield Road,53.339005,-6.300217,NaN,30
3412088,dublin_bikes,2026-06-30 23:55:00,92,40,0,True,True,True,HEUSTON BRIDGE (NORTH),NaN,Heuston Bridge (North),53.347800,-6.292432,NaN,40


Inspecionando os dados

In [3]:
df_completo.shape

(3412090, 15)

In [4]:
df_completo.info()

<class 'pandas.DataFrame'>
RangeIndex: 3412090 entries, 0 to 3412089
Data columns (total 15 columns):
 #   Column               Dtype  
---  ------               -----  
 0   system_id            str    
 1   last_reported        str    
 2   station_id           int64  
 3   num_bikes_available  int64  
 4   num_docks_available  int64  
 5   is_installed         bool   
 6   is_renting           bool   
 7   is_returning         bool   
 8   name                 str    
 9   short_name           float64
 10  address              str    
 11  lat                  float64
 12  lon                  float64
 13  region_id            float64
 14  capacity             int64  
dtypes: bool(3), float64(4), int64(4), str(4)
memory usage: 322.1 MB


In [5]:
df_completo.isnull().sum().sort_values(ascending=False)

short_name             3412090
region_id              3412090
system_id                    0
last_reported                0
station_id                   0
num_bikes_available          0
num_docks_available          0
is_installed                 0
is_renting                   0
is_returning                 0
name                         0
address                      0
lat                          0
lon                          0
capacity                     0
dtype: int64

In [6]:
df_completo.duplicated().sum()

np.int64(0)

Removendo colunas nulas

In [7]:
df_completo = df_completo.drop(columns=["short_name", "region_id"])

In [8]:
df_completo.shape

(3412090, 13)

Convertendo tipo string de last_reported para date

In [9]:
df_completo["last_reported"] = pd.to_datetime(df_completo["last_reported"])

In [10]:
df_completo.info()

<class 'pandas.DataFrame'>
RangeIndex: 3412090 entries, 0 to 3412089
Data columns (total 13 columns):
 #   Column               Dtype         
---  ------               -----         
 0   system_id            str           
 1   last_reported        datetime64[us]
 2   station_id           int64         
 3   num_bikes_available  int64         
 4   num_docks_available  int64         
 5   is_installed         bool          
 6   is_renting           bool          
 7   is_returning         bool          
 8   name                 str           
 9   address              str           
 10  lat                  float64       
 11  lon                  float64       
 12  capacity             int64         
dtypes: bool(3), datetime64[us](1), float64(2), int64(4), str(3)
memory usage: 270.1 MB


Contando valores distintos por coluna

In [11]:
df_completo.nunique()

system_id                  1
last_reported          48580
station_id               115
num_bikes_available       41
num_docks_available       42
is_installed               2
is_renting                 2
is_returning               2
name                     116
address                  116
lat                      132
lon                      132
capacity                  17
dtype: int64

Removendo colunas sem informação

In [12]:
df_completo = df_completo.drop(columns=["system_id"])

In [13]:
df_completo.shape

(3412090, 12)

Conferindo quais estações mudaram de nome, 115 estações para 116 nomes 

In [14]:
names = df_completo.groupby('station_id')['name'].nunique()
print(names[names > 1])

station_id
34    2
Name: name, dtype: int64


In [15]:
df_34 = df_completo['station_id'] == 34
df_completo[df_34]['name'].unique()

<StringArray>
['PORTOBELLO HARBOUR', 'LENNOX STREET']
Length: 2, dtype: str

In [16]:
df_completo[df_34].groupby('name')['last_reported'].agg(["min", "max"])

,min,max
name,,
LENNOX STREET,2026-02-12 14:05:00,2026-06-30 23:55:00
PORTOBELLO HARBOUR,2026-01-01 00:05:00,2026-01-13 10:55:00


Estação 34 trocou de nome no meio da série (Portobello Harbour → Lennox Street)

115 estações e 132 coordenadas

In [17]:
coordenadas = df_completo.groupby('station_id')['lat'].nunique()
print(coordenadas[coordenadas > 1])

station_id
5     2
7     2
18    2
19    2
20    2
21    2
23    2
24    2
26    2
27    2
29    2
34    2
36    2
38    2
41    2
44    2
56    2
Name: lat, dtype: int64


Não precisamos de coodernadas para o projeto

In [18]:
df_completo = df_completo.drop(columns=['lat', 'lon'])

Conferindo se bikes + docks bate com capacity

In [19]:
inconsistentes = df_completo['num_bikes_available'] + df_completo['num_docks_available'] != df_completo['capacity']
inconsistentes.sum()

np.int64(151573)

Vendo o tamanho da diferença nas linhas que não batem

In [20]:
diferenca = df_completo['capacity'] - (df_completo['num_bikes_available'] + df_completo['num_docks_available'])
diferenca[inconsistentes].value_counts()

 1     94995
 20    23325
 2     15494
 3      4447
 40     3993
 7      3354
 10     1485
 4      1468
 30      615
 5       592
 18      313
 21      311
 38      134
 6       126
 29       94
 39       88
-1        88
 8        82
 28       75
 11       73
 19       38
 22       38
 26       32
 27       31
 37       28
 14       27
 25       27
 33       25
 35       23
 34       22
 17       18
 13       17
 32       15
 23       15
 24       12
 36       11
 15       10
-2         9
 16        7
 12        6
 31        5
-3         5
Name: count, dtype: int64

Olhando as linhas com diferença de 20

In [21]:
df_completo[diferenca == 20][['num_bikes_available', 'num_docks_available']].head()

,num_bikes_available,num_docks_available
2319,8,2
20437,8,2
39421,5,0
58085,5,0
96123,5,0


Vendo se a diferença se concentra em alguma estação

In [22]:
df_completo[diferenca >= 10]['station_id'].value_counts()

station_id
51     26193
34      1511
30       355
25       329
26       292
       ...  
33         1
16         1
109        1
94         1
106        1
Name: count, Length: 88, dtype: int64

Removendo capacity, que não bate com o total reportado, a ocupação vai ser calculada sobre bikes + docks de cada linha

In [23]:
df_completo =df_completo.drop(columns = 'capacity')

Conferindo se existe valor negativo

In [24]:
(df_completo['num_bikes_available'] < 0).sum()

np.int64(0)

Olhando as três colunas de status

In [25]:
df_completo[['is_installed', 'is_renting', 'is_returning']].sum()

is_installed    3408656
is_renting      3408656
is_returning    3408656
dtype: int64

Mantendo só as linhas em que a estação estava em operação

In [26]:
df_completo = df_completo[df_completo['is_renting'] == 1]

Removendo as colunas de status, agora constantes

In [27]:
df_completo = df_completo.drop(columns = ['is_renting', 'is_returning', 'is_installed'])

Conferindo o resultado final

In [28]:
df_completo.shape

(3408656, 6)

Salvando a base limpa em data/processed

In [29]:
df_completo.to_csv("../data/processed/dublin_bikes_clean.csv", index=False)